# MrScraper Price Intelligence

This notebook is the concise report for the price reconstruction pipeline. The production code lives in `src/`, the runnable entrypoints live in `scripts/`, and the detailed exploratory checks live in `eda.ipynb`. The best results from validation points to using recent same-entity prices as the primary predictor and using anchor-gated calibration only for fallback rows. Validation shows product/entity history is substantially stronger than the global marketplace model when prior observations exist.


## How To Run

Full validation:

```bash
.venv/bin/python scripts/run_validation.py
```

Final test prediction:

```bash
.venv/bin/python scripts/predict_test.py
```

Fast smoke checks:

```bash
.venv/bin/python scripts/run_validation.py --sample-rows 5000 --validation-days 1 --anchors-per-day 20 --iterations 5 --output-dir outputs/smoke_validation
.venv/bin/python scripts/predict_test.py --iterations 5 --output-dir outputs/smoke_prediction
```


In [1]:
from pathlib import Path
from dataclasses import replace
import sys

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))

from src.config import PipelineConfig
from src.data import load_data
from src.features import build_features, get_cat_feature_indices, get_feature_columns
from src.inference import run_prediction
from src.validation import run_outage_validation

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

config = PipelineConfig()

FINAL_VARIANT = "anchor_gated_fallback_calibration"
FINAL_VALIDATION_LABEL = "anchor_gated_fallback_calibration_entity_blend_no_calibration"

def tidy_metrics(df, model_col="base_model"):
    out = df.copy()
    if model_col in out.columns:
        out = out.rename(columns={model_col: "variant"})
    metric_cols = [col for col in ["MAE", "RMSE", "MAPE"] if col in out.columns]
    return out.sort_values(metric_cols[0] if metric_cols else out.columns[0]).reset_index(drop=True)

def show_metric_table(df, title=None, model_col="base_model"):
    if title:
        print(title)
    display(tidy_metrics(df, model_col=model_col))

config


PipelineConfig(train_path=PosixPath('ecommerce_price_prediction-train.csv'), test_path=PosixPath('ecommerce_price_prediction-test-3-days.csv'), output_dir=PosixPath('outputs'), target='price', date_col='capturedAt', random_seed=42, validation_days=3, anchors_per_day=100, validation_test_like_only=False, validation_mask_hidden_like_test=True, model_hidden_available_features_only=True, entity_smoothing=20.0, calibration_smoothing=8.0, calibration_delta_cap=None, selective_calibration=False, calibration_min_anchors=20, calibration_max_residual_iqr=None, calibration_min_abs_global_delta=0.0, second_stage_alpha=10.0, second_stage_min_anchors=10, include_experimental_variants=False, prediction_variant='anchor_gated_fallback_calibration', catboost_iterations=1200, catboost_learning_rate=0.05, catboost_depth=8, id_cols=['shopId', 'itemId', 'modelId'], cat_cols=['shopId', 'itemId', 'modelId', 'promotionId', 'cat_id', 'brand'], bool_cols=['is_free_shipping', 'is_pre_order', 'is_official_shop', '

## Data Overview

The train file contains historical prices. The test file contains rows from the outage window, where most `price` values are missing and the non-missing prices act as same-day anchors.


In [2]:
train_raw, test_raw = load_data(config.train_path, config.test_path)

summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_raw),
            "columns": train_raw.shape[1],
            "min_capturedAt": train_raw[config.date_col].min(),
            "max_capturedAt": train_raw[config.date_col].max(),
            "missing_price": train_raw[config.target].isna().sum(),
        },
        {
            "dataset": "test",
            "rows": len(test_raw),
            "columns": test_raw.shape[1],
            "min_capturedAt": test_raw[config.date_col].min(),
            "max_capturedAt": test_raw[config.date_col].max(),
            "missing_price": test_raw[config.target].isna().sum(),
        },
    ]
)
summary


,dataset,rows,columns,min_capturedAt,max_capturedAt,missing_price
0,train,306226,26,2025-01-01 22:37:36.424,2025-03-22 04:27:05.302,0
1,test,25900,26,2025-03-22 04:27:05.325,2025-03-24 23:30:04.453,25600


In [3]:
train_raw.head()

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,brand,is_free_shipping,is_pre_order,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-01-01 22:37:36.424,1008004,29302587134,185168612932,41900000,68000000,534463751667712,100630,NaN,NaN,38,38,MlRAE 未來美,f,f,41900000,41900000,0.0000,0,0,4.9752,55.0000,158,f,f,f
1,2025-01-02 01:45:48.634,1006671485,18087284916,166287259494,16900000,0,0,100013,NaN,NaN,14,14,NaN,f,f,11900000,14900000,5.0000,1,1,4.9364,70.0000,3955,t,f,f
2,2025-01-02 01:45:48.634,1006671485,18087284916,166287259497,11900000,13900000,474465164066816,100013,NaN,NaN,14,14,NaN,f,f,11900000,14900000,5.0000,1,1,4.9364,70.0000,3955,t,f,f
3,2025-01-02 01:45:48.634,1006671485,18087284916,166287259492,16900000,0,0,100013,NaN,NaN,14,14,NaN,f,f,11900000,14900000,5.0000,1,1,4.9364,70.0000,3955,t,f,f
4,2025-01-02 01:45:48.634,1006671485,18087284916,166287259490,16900000,0,0,100013,NaN,NaN,14,14,NaN,f,f,11900000,14900000,5.0000,1,1,4.9364,70.0000,3955,t,f,f


In [4]:
test_raw.head()

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,brand,is_free_shipping,is_pre_order,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pipeline Design

The pipeline is intentionally split into simple stages:

1. Create normal row-level features.
2. Create leakage-safe historical aggregate features.
3. Train a global CatBoost model on `log1p(price)`.
4. Predict the outage rows in log-price space.
5. Blend the model prediction with an entity historical median prior.
6. Use anchor residuals to calibrate predictions by day.
7. Fill missing prices and preserve known anchor prices.

This structure matches the test setup: anchors are available at prediction time, so they can be used for calibration without leaking hidden prices.


## Feature Engineering

### Row-Level Features

The row-level features come from the current row only:

- time: `hour`, `dayofweek`, `day`, `month`, `is_weekend`
- discount: `has_discount`, `discount_ratio`, `price_before_log`, `raw_discount_log`, `show_discount_ratio`
- stock: `stock_ratio`, `stock_gap`, `is_low_stock`
- engagement: `total_rating_count_log`, `cmt_count_log`, `shop_follower_count_log`, `review_strength`, `shop_strength`
- categorical IDs: `shopId`, `itemId`, `modelId`, `promotionId`, `cat_id`, `brand`
- booleans: free shipping, pre-order, official/verified/preferred seller flags

### Historical Features

For each of `modelId`, `itemId`, `shopId`, `cat_id`, and `brand`, the pipeline computes historical log-price statistics:

- count
- mean
- median
- standard deviation
- minimum
- maximum

It also creates two fallback features:

- recent-price features for `modelId` and `itemId`: last observed price, last 3-observation median, last 7-observation median, prior count, and hours since last seen.
- `fallback_entity_price_log`: most specific available recent/entity prior, falling back from `modelId` last price to `itemId` last price, then historical medians from broader groups.
- `fallback_entity_history_count`: the matching amount of historical evidence.


In [5]:
# Build features on a small chronological sample for inspection only.
# Full training uses the complete dataset through the scripts.
inspect_history = train_raw.sort_values(config.date_col).head(10000)
inspect_features = build_features(inspect_history, inspect_history.head(1000), config)
feature_cols = get_feature_columns(inspect_features, config)
cat_idx = get_cat_feature_indices(inspect_features, feature_cols, config)

pd.DataFrame(
    {
        "n_features": [len(feature_cols)],
        "n_categorical_features": [len(cat_idx)],
    }
)


,n_features,n_categorical_features
0,51,5


In [6]:
historical_feature_preview = [
    col for col in inspect_features.columns
    if col.endswith("_log") or col.endswith("_count") or col.startswith("fallback_entity")
]

pd.Series(historical_feature_preview, name="historical_or_log_feature").head(40)


0                  total_rating_count
1                           cmt_count
2                 shop_follower_count
3                    price_before_log
4                    raw_discount_log
5              total_rating_count_log
6                       cmt_count_log
7             shop_follower_count_log
8          modelId_recent_prior_count
9              modelId_last_price_log
10    modelId_last_3_median_price_log
11    modelId_last_7_median_price_log
12          itemId_recent_prior_count
13              itemId_last_price_log
14     itemId_last_3_median_price_log
15     itemId_last_7_median_price_log
16                modelId_price_count
17             modelId_price_mean_log
18           modelId_price_median_log
19              modelId_price_std_log
20              modelId_price_min_log
21              modelId_price_max_log
22                 itemId_price_count
23              itemId_price_mean_log
24            itemId_price_median_log
25               itemId_price_std_log
26          

## Entity Blending

The model already receives historical features, but the pipeline also applies an explicit shrinkage blend after prediction:

```text
entity_weight = count / (count + entity_smoothing)
blended_pred_log =
    (1 - entity_weight) * model_pred_log
    + entity_weight * fallback_entity_price_log
```

The default `entity_smoothing` is `20`. That means 20 historical observations gives the entity prior 50% weight. This is a tunable hyperparameter, not a fixed truth.

The justification is that exact product variants are often price-stable. The criticism is that this is partly redundant with model features and should be validated or tuned.


## Anchor Calibration

Calibration uses the same-day anchor prices. For anchor rows:

```text
residual_log = log1p(anchor_price) - predicted_log_price
```

The median residual is used as a day-level correction. The pipeline also estimates residuals by `cat_id`, `shopId`, `brand`, and `promotionId`, with shrinkage toward the global day residual:

```text
weight = anchor_count / (anchor_count + calibration_smoothing)
```

The default `calibration_smoothing` is `8`. This keeps small anchor groups from overcorrecting the rest of the day.


## Anchor-Set Utilization Variants

The current recommended strategy is intentionally simple: use recent same-entity prices first, then fall back to the trained model only when recent entity history is missing.

1. **Last-price baseline**
   Use the recent entity prior directly, without CatBoost. This tests the anchor EDA finding that same-`modelId` last price is extremely strong.

2. **Hybrid last-price + uncalibrated fallback model**
   `hybrid_last_price_uncalibrated_fallback` is the primary candidate. It uses recent entity price when available, otherwise falls back to the entity-blend model.

3. **Hybrid last-price + calibrated fallback**
   `hybrid_last_price_calibrated_fallback` is the main anchor-use alternative. It keeps strong last-price rows untouched and applies anchor calibration only to fallback rows.

4. **Global / segment calibration**
   Estimate anchor residuals and apply a day-level plus segment-level correction. These variants remain useful as baselines for checking whether calibration is actually helping.

5. **Model selection using anchor error**
   Score candidate variants on anchor rows, choose the lowest-anchor-MAE variant for that day, and use it for hidden rows. This appears as `anchor_model_selected_*`.

`second_stage_residual` is included in validation as a learned anchor-residual diagnostic. It fits a small same-day Ridge model on anchor residual features, then predicts a log-price correction for hidden rows.


## Validation Methodology

Validation mirrors the outage setting rather than using a random row split:

1. Hold out recent historical days.
2. Train only on earlier history.
3. Sample anchor rows from each held-out day.
4. Hide the remaining prices and blank the same fields that are blank in the real test file.
5. Recover only leakage-safe historical metadata and recent price priors.
6. Evaluate hidden rows with MAE, RMSE, and MAPE in raw price space.

The model predicts `log1p(price)` internally. Metrics are computed after converting predictions back with `expm1`. Segment outputs break metrics down by date, price bucket, history-count bucket, and calibration status.


In [7]:
# Optional: run outage-style validation from inside the notebook.
# For normal review, prefer loading saved outputs in the next cell.
import importlib

# Reload dependencies first so long-lived notebook kernels do not keep stale symbols.
import src.strategies as strategies_module
import src.features as features_module
import src.calibration as calibration_module
import src.model as model_module
import src.validation as validation_module

strategies_module = importlib.reload(strategies_module)
features_module = importlib.reload(features_module)
calibration_module = importlib.reload(calibration_module)
model_module = importlib.reload(model_module)
validation_module = importlib.reload(validation_module)
run_outage_validation = validation_module.run_outage_validation

RUN_VALIDATION = True
VALIDATION_TEST_LIKE_ONLY = True
VALIDATION_ITERATIONS = config.catboost_iterations
VALIDATION_OUTPUT_DIR = config.output_dir

if RUN_VALIDATION:
    validation_config = replace(
        config,
        output_dir=VALIDATION_OUTPUT_DIR,
        catboost_iterations=VALIDATION_ITERATIONS,
        validation_test_like_only=VALIDATION_TEST_LIKE_ONLY,
    )
    validation_results, validation_summary, validation_segments, validation_predictions = run_outage_validation(
        train_raw, validation_config
    )

    validation_config.output_dir.mkdir(exist_ok=True)
    validation_results.to_csv(validation_config.output_dir / "validation_results.csv", index=False)
    validation_summary.to_csv(validation_config.output_dir / "validation_summary.csv")
    validation_segments.to_csv(validation_config.output_dir / "validation_segments.csv", index=False)
    validation_predictions.to_csv(validation_config.output_dir / "validation_predictions.csv", index=False)

    show_metric_table(validation_summary.reset_index(), "Validation summary")
else:
    print("RUN_VALIDATION is False. Loading saved outputs in the next cell.")


0:	learn: 0.9876903	total: 264ms	remaining: 5m 16s
200:	learn: 0.0368925	total: 35.9s	remaining: 2m 58s
400:	learn: 0.0298229	total: 1m 14s	remaining: 2m 29s
600:	learn: 0.0255709	total: 1m 55s	remaining: 1m 55s
800:	learn: 0.0229817	total: 2m 35s	remaining: 1m 17s
1000:	learn: 0.0212531	total: 3m 15s	remaining: 38.8s
1199:	learn: 0.0200154	total: 3m 52s	remaining: 0us
Validation summary


,variant,MAE,RMSE,MAPE
0,anchor_gated_fallback_calibration_entity_blend...,"38,129.1479","959,527.0612",0.1424
1,anchor_model_selected_last_price_baseline,"38,129.1479","959,527.0612",0.1424
2,hybrid_last_price_calibrated_fallback,"38,129.1479","959,527.0612",0.1424
3,hybrid_last_price_uncalibrated_fallback,"38,129.1479","959,527.0612",0.1424
4,last_price_baseline,"38,129.1479","959,527.0612",0.1424
5,entity_blend_no_calibration,"349,680.3851","3,879,856.3544",0.4616
6,entity_blend_calibrated,"369,682.7463","3,758,224.1094",0.4630
7,second_stage_residual,"468,972.7530","4,115,702.4204",0.5588
8,global_no_calibration,"948,706.2455","9,769,068.5750",1.2067
9,global_calibrated,"968,181.7244","9,568,990.0772",1.2003


In [8]:
# Load saved validation outputs and display report-ready tables.
validation_results_path = config.output_dir / "validation_results.csv"
validation_summary_path = config.output_dir / "validation_summary.csv"
validation_segments_path = config.output_dir / "validation_segments.csv"

if not validation_summary_path.exists():
    raise FileNotFoundError(f"Missing {validation_summary_path}. Run scripts/run_validation.py first.")

saved_validation_summary = pd.read_csv(validation_summary_path)
show_metric_table(saved_validation_summary, "Overall hidden-row validation metrics")

approach_map = {
    "global_no_calibration": "Approach 1: global",
    "global_calibrated": "Approach 1: global + anchors",
    "entity_blend_no_calibration": "Approach 2: entity blend",
    "entity_blend_calibrated": "Approach 2: entity blend + anchors",
    "last_price_baseline": "Entity last-price baseline",
    "hybrid_last_price_uncalibrated_fallback": "Final family: recent entity + uncalibrated fallback",
    "hybrid_last_price_calibrated_fallback": "Final family: calibrated fallback",
    "anchor_gated_fallback_calibration_entity_blend_no_calibration": "Final default: anchor-gated fallback",
    "second_stage_residual": "Diagnostic: learned anchor residual",
}
approach_comparison = saved_validation_summary.copy()
approach_comparison["approach"] = approach_comparison["base_model"].map(approach_map).fillna("Anchor-selected variant")
approach_comparison = approach_comparison[["approach", "base_model", "MAE", "RMSE", "MAPE"]].sort_values("MAE")
display(approach_comparison)

if validation_segments_path.exists():
    saved_validation_segments = pd.read_csv(validation_segments_path)
    final_segments = saved_validation_segments.query("variant == @FINAL_VALIDATION_LABEL")

    print("Final strategy by validation date")
    display(final_segments.query("segment == 'date'").sort_values("segment_value"))

    print("Final strategy by price bucket")
    display(final_segments.query("segment == 'price_bucket'").sort_values("segment_value"))

    print("Final strategy by history-count bucket")
    display(final_segments.query("segment == 'history_count_bucket'").sort_values("segment_value"))
else:
    print(f"Missing {validation_segments_path}. Segment tables are unavailable.")


Overall hidden-row validation metrics


,variant,MAE,RMSE,MAPE
0,anchor_gated_fallback_calibration_entity_blend...,"38,129.1479","959,527.0612",0.1424
1,anchor_model_selected_last_price_baseline,"38,129.1479","959,527.0612",0.1424
2,hybrid_last_price_calibrated_fallback,"38,129.1479","959,527.0612",0.1424
3,hybrid_last_price_uncalibrated_fallback,"38,129.1479","959,527.0612",0.1424
4,last_price_baseline,"38,129.1479","959,527.0612",0.1424
5,entity_blend_no_calibration,"349,680.3851","3,879,856.3544",0.4616
6,entity_blend_calibrated,"369,682.7463","3,758,224.1094",0.4630
7,second_stage_residual,"468,972.7530","4,115,702.4204",0.5588
8,global_no_calibration,"948,706.2455","9,769,068.5750",1.2067
9,global_calibrated,"968,181.7244","9,568,990.0772",1.2003


,approach,base_model,MAE,RMSE,MAPE
0,Final default: anchor-gated fallback,anchor_gated_fallback_calibration_entity_blend...,"38,129.1479","959,527.0612",0.1424
1,Anchor-selected variant,anchor_model_selected_last_price_baseline,"38,129.1479","959,527.0612",0.1424
2,Final family: calibrated fallback,hybrid_last_price_calibrated_fallback,"38,129.1479","959,527.0612",0.1424
3,Final family: recent entity + uncalibrated fal...,hybrid_last_price_uncalibrated_fallback,"38,129.1479","959,527.0612",0.1424
4,Entity last-price baseline,last_price_baseline,"38,129.1479","959,527.0612",0.1424
5,Approach 2: entity blend,entity_blend_no_calibration,"349,680.3851","3,879,856.3544",0.4616
6,Approach 2: entity blend + anchors,entity_blend_calibrated,"369,682.7463","3,758,224.1094",0.4630
7,Diagnostic: learned anchor residual,second_stage_residual,"468,972.7530","4,115,702.4204",0.5588
8,Approach 1: global,global_no_calibration,"948,706.2455","9,769,068.5750",1.2067
9,Approach 1: global + anchors,global_calibrated,"968,181.7244","9,568,990.0772",1.2003


Final strategy by validation date


,segment,segment_value,variant,MAE,RMSE,MAPE,n_eval
0,date,2025-03-20,anchor_gated_fallback_calibration_entity_blend...,"17,203.3469","557,276.0013",0.0427,7888
1,date,2025-03-21,anchor_gated_fallback_calibration_entity_blend...,"71,138.4693","1,410,810.3472",0.2120,7879
2,date,2025-03-22,anchor_gated_fallback_calibration_entity_blend...,"26,045.6274","910,494.8352",0.1725,4734


Final strategy by price bucket


,segment,segment_value,variant,MAE,RMSE,MAPE,n_eval
30,price_bucket,price_q1,anchor_gated_fallback_calibration_entity_blend...,595.9032,"16,934.5327",0.0099,5370
31,price_bucket,price_q2,anchor_gated_fallback_calibration_entity_blend...,"39,939.6378","1,146,278.2046",0.2868,4970
32,price_bucket,price_q3,anchor_gated_fallback_calibration_entity_blend...,"32,339.5854","622,785.5142",0.1311,5065
33,price_bucket,price_q4,anchor_gated_fallback_calibration_entity_blend...,"89,089.4819","1,631,229.7564",0.1337,5096


Final strategy by history-count bucket


,segment,segment_value,variant,MAE,RMSE,MAPE,n_eval
70,history_count_bucket,1-5,anchor_gated_fallback_calibration_entity_blend...,"72,636.8159","494,647.9727",0.4992,201
71,history_count_bucket,100+,anchor_gated_fallback_calibration_entity_blend...,"69,873.7988","1,478,633.6705",0.2601,8637
72,history_count_bucket,21-100,anchor_gated_fallback_calibration_entity_blend...,"16,416.8451","505,620.4953",0.0395,9807
73,history_count_bucket,6-20,anchor_gated_fallback_calibration_entity_blend...,"21,767.2414","576,329.0487",0.0480,1856


## Hyperparameter Tuning

The pipeline includes a tuning script that searches both model hyperparameters and post-processing constants:

- CatBoost `learning_rate`
- CatBoost `depth`
- CatBoost `iterations`
- `entity_smoothing` for the entity prior blend
- `calibration_smoothing` for anchor group shrinkage
This is important because the original `20` and `8` smoothing constants were reasonable defaults, not learned values. Calibration caps and non-MAE selection metrics are available in the CLI as experimental options, but they are disabled in the default workflow for now.

Example smoke command:

```bash
.venv/bin/python scripts/tune_hyperparameters.py \
  --sample-rows 5000 \
  --validation-days 1 \
  --anchors-per-day 20 \
  --learning-rates 0.05 \
  --depths 6 \
  --iterations 5 \
  --entity-smoothing 10,20 \
  --calibration-smoothing 4,8 \
  --output-dir outputs/smoke_tuning
```


In [9]:
tuning_summary_path = config.output_dir / "tuning" / "tuning_summary.csv"
tuning_details_path = config.output_dir / "tuning" / "tuning_details.csv"

if tuning_summary_path.exists():
    tuning_summary = pd.read_csv(tuning_summary_path)
    display(tuning_summary.head(20))
else:
    print(f"Missing {tuning_summary_path}. Run scripts/tune_hyperparameters.py first.")

if tuning_details_path.exists():
    tuning_details = pd.read_csv(tuning_details_path)
    print("Detailed tuning rows:", len(tuning_details))
else:
    print(f"Missing {tuning_details_path}. Run scripts/tune_hyperparameters.py first.")


Missing outputs/tuning/tuning_summary.csv. Run scripts/tune_hyperparameters.py first.
Missing outputs/tuning/tuning_details.csv. Run scripts/tune_hyperparameters.py first.


## Prediction Output

The prediction script trains on the full training file, predicts the provided test file, fills only missing prices, and verifies that known anchor prices are preserved. The default variant is `anchor_gated_fallback_calibration`.


In [10]:
completed_path = config.output_dir / "completed_test_predictions.csv"

if completed_path.exists():
    completed = pd.read_csv(completed_path)
    original_test = pd.read_csv(config.test_path)
    anchor_mask = original_test[config.target].notna()
    prediction_check = pd.DataFrame([
        {"check": "rows", "value": len(completed)},
        {"check": "missing_prices", "value": completed[config.target].isna().sum()},
        {"check": "known_anchor_rows", "value": int(anchor_mask.sum())},
        {"check": "min_predicted_price", "value": completed[config.target].min()},
        {"check": "median_predicted_price", "value": completed[config.target].median()},
        {"check": "max_predicted_price", "value": completed[config.target].max()},
    ])
    display(prediction_check)
    display(completed.head(10))
else:
    print(f"Missing {completed_path}. Run scripts/predict_test.py first.")


,check,value
0,rows,"25,900.0000"
1,missing_prices,0.0000
2,known_anchor_rows,300.0000
3,min_predicted_price,"100,000.0000"
4,median_predicted_price,"15,500,000.0000"
5,max_predicted_price,"1,660,000,000.0000"


,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,brand,is_free_shipping,is_pre_order,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-03-22 04:27:05.325,1009757562,27904817781,241162534605,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-03-22 04:27:05.325,1009757562,27904817781,241162534605,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-03-22 04:27:05.388,1009757562,22983470729,194654575543,27800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-03-22 04:27:05.388,1009757562,22983470729,226663185840,49500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Optional: run prediction from inside the notebook.
# This can take time with the default 1200 CatBoost iterations.
RUN_FULL_PREDICTION = True

if RUN_FULL_PREDICTION:
    submission = run_prediction(config)
    display(submission.head())


0:	learn: 0.9865903	total: 201ms	remaining: 4m
200:	learn: 0.0388651	total: 39.9s	remaining: 3m 18s
400:	learn: 0.0304973	total: 1m 28s	remaining: 2m 56s
600:	learn: 0.0264517	total: 2m 7s	remaining: 2m 6s
800:	learn: 0.0237994	total: 2m 47s	remaining: 1m 23s
1000:	learn: 0.0219920	total: 3m 39s	remaining: 43.7s
1199:	learn: 0.0204928	total: 4m 29s	remaining: 0us


,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,raw_discount,show_discount,brand,is_free_shipping,is_pre_order,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8400000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,7500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
